# OctoTetrahedral AGI — ARC Prize 2026 ARC-AGI-2 Submission

Mega solver with 24 strategies + compound cascade.

In [ ]:
import os, json, time, signal, numpy as np
from pathlib import Path
import glob

OUT_FILE = Path('/kaggle/working/submission.json')
TASK_TIMEOUT = 30  # seconds per task

# Auto-discover the test challenges file under /kaggle/input/
_candidates = glob.glob('/kaggle/input/**/arc-agi_test_challenges.json', recursive=True)
if not _candidates:
    # Show what's available for debugging
    print("Available input dirs:", os.listdir('/kaggle/input/'))
    for d in os.listdir('/kaggle/input/'):
        print(f"  /kaggle/input/{d}:", os.listdir(f'/kaggle/input/{d}'))
    raise FileNotFoundError("Cannot find arc-agi_test_challenges.json in /kaggle/input/")

TEST_FILE = Path(_candidates[0])
DATA_DIR = TEST_FILE.parent
print(f"Using data: {TEST_FILE}")
print(f"All files: {list(DATA_DIR.iterdir())}")

In [ ]:
#!/usr/bin/env python3
"""
MEGA ARC SOLVER: combines ALL solver strategies into one runner.
Runs every strategy on every task, reports combined score.
"""

import os, sys, json, time, multiprocessing
import numpy as np
from collections import Counter, defaultdict
from scipy import ndimage


def grids_match(a, b):
    a, b = np.array(a), np.array(b)
    return a.shape == b.shape and np.array_equal(a, b)

def find_bg(grid):
    vals, counts = np.unique(grid, return_counts=True)
    return int(vals[counts.argmax()])


# ═══════════════════════════════════════════════
# Neighborhood LUT
# ═══════════════════════════════════════════════

def get_nb(grid, r, c, radius=1):
    h, w = grid.shape
    vals = []
    for dr in range(-radius, radius+1):
        for dc in range(-radius, radius+1):
            nr, nc = r+dr, c+dc
            vals.append(int(grid[nr, nc]) if 0 <= nr < h and 0 <= nc < w else -1)
    return tuple(vals)


def solve_nb_lut(task, radius=1):
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    lut = {}
    for ex in train:
        inp, out = np.array(ex['input']), np.array(ex['output'])
        h, w = inp.shape
        for r in range(h):
            for c in range(w):
                key = get_nb(inp, r, c, radius)
                val = int(out[r, c])
                if key in lut and lut[key] != val:
                    return None
                lut[key] = val
    
    test_inp = np.array(task['test'][0]['input'])
    h, w = test_inp.shape
    result = np.zeros((h, w), dtype=int)
    for r in range(h):
        for c in range(w):
            key = get_nb(test_inp, r, c, radius)
            if key not in lut:
                return None
            result[r, c] = lut[key]
    
    return result.tolist(), f'nb_lut_r{radius}'


# ═══════════════════════════════════════════════
# Abstract Features (expanded)
# ═══════════════════════════════════════════════

def get_features(grid, r, c):
    h, w = grid.shape
    val = int(grid[r, c])
    bg = find_bg(grid)
    
    n4 = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < h and 0 <= nc < w:
            n4.append(int(grid[nr, nc]))
    n8 = list(n4)
    for dr, dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < h and 0 <= nc < w:
            n8.append(int(grid[nr, nc]))
    
    nn_bg = sum(1 for n in n8 if n != bg)
    nn4_bg = sum(1 for n in n4 if n != bg)
    adj_same = sum(1 for n in n4 if n == val)
    adj_colors = tuple(sorted(set(n4)))
    
    feats = [
        ('v', val),
        ('v_nn', val, nn_bg),
        ('v_nc', val, adj_colors),
        ('v_border', val, r==0 or r==h-1 or c==0 or c==w-1),
        ('v_adj_same', val, adj_same),
        ('v_maj', val, Counter(n8).most_common(1)[0][0] if n8 else bg),
        ('v_parity', val, r%2, c%2),
        ('v_bg_adj', val, val==bg, nn4_bg),
        # New features
        ('v_row', val, r),
        ('v_col', val, c),
        ('v_rc', val, r, c),
        ('v_nn4', val, nn4_bg),
        ('v_rmod3', val, r%3, c%3),
        ('v_diag', val, (r+c)%2),
        ('v_diag2', val, (r-c)%3 if r>=c else (c-r)%3),
        ('v_dist_edge', val, min(r, c, h-1-r, w-1-c)),
        ('v_rmod', val, r % max(1, h//2)),
        ('v_cmod', val, c % max(1, w//2)),
        ('n4_sorted', tuple(sorted(n4))),
        ('n4_val', val, tuple(sorted(n4))),
        ('v_adj_diff', val, tuple(sorted(set(n4) - {val}))),
        # Paired features
        ('v_rowu_colu', val, len(set(int(x) for x in grid[r, :])), len(set(int(x) for x in grid[:, c]))),
        ('v_isbg_rcparity', val, int(val == bg), (r + c) % 2),
        ('v_isbg_n4nonbg', val, int(val == bg), nn4_bg),
    ]
    return feats


def solve_abstract(task):
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    n_feats = len(get_features(np.array(train[0]['input']), 0, 0))
    
    for fi in range(n_feats):
        lut = {}
        ok = True
        for ex in train:
            inp, out = np.array(ex['input']), np.array(ex['output'])
            h, w = inp.shape
            for r in range(h):
                for c in range(w):
                    key = get_features(inp, r, c)[fi]
                    val = int(out[r, c])
                    if key in lut and lut[key] != val:
                        ok = False
                        break
                    lut[key] = val
                if not ok:
                    break
            if not ok:
                break
        
        if ok:
            test_inp = np.array(task['test'][0]['input'])
            h, w = test_inp.shape
            result = np.zeros((h, w), dtype=int)
            complete = True
            for r in range(h):
                for c in range(w):
                    key = get_features(test_inp, r, c)[fi]
                    if key not in lut:
                        complete = False
                        break
                    result[r, c] = lut[key]
                if not complete:
                    break
            if complete:
                return result.tolist(), f'abstract_f{fi}'
    
    return None


# ═══════════════════════════════════════════════
# Grid Partition (separator detection)
# ═══════════════════════════════════════════════

def find_seps(grid, bg=0):
    h, w = grid.shape
    srows = [(r, int(grid[r,0])) for r in range(h) 
             if len(set(grid[r].flatten().tolist())) == 1 and int(grid[r,0]) != bg]
    scols = [(c, int(grid[0,c])) for c in range(w) 
             if len(set(grid[:,c].flatten().tolist())) == 1 and int(grid[0,c]) != bg]
    return srows, scols


def get_subgrids(grid, srows, scols):
    h, w = grid.shape
    rows = [-1] + [r for r, _ in srows] + [h]
    cols = [-1] + [c for c, _ in scols] + [w]
    sgs = []
    for i in range(len(rows)-1):
        for j in range(len(cols)-1):
            r1, r2 = rows[i]+1, rows[i+1]
            c1, c2 = cols[j]+1, cols[j+1]
            if r1 < r2 and c1 < c2:
                sgs.append(((r1, c1, r2, c2), grid[r1:r2, c1:c2].copy()))
    return sgs


def solve_partition(task):
    train = task['train']
    inp0 = np.array(train[0]['input'])
    out0 = np.array(train[0]['output'])
    bg = find_bg(inp0)
    
    sr, sc = find_seps(inp0, bg)
    if not sr and not sc:
        return None
    
    sgs0 = get_subgrids(inp0, sr, sc)
    if not sgs0:
        return None
    
    # Try: output = specific subgrid by index
    for idx in range(len(sgs0)):
        if grids_match(sgs0[idx][1], out0):
            ok = True
            for ex in train[1:]:
                inp = np.array(ex['input'])
                out = np.array(ex['output'])
                si, sci = find_seps(inp, find_bg(inp))
                sgs = get_subgrids(inp, si, sci)
                if idx >= len(sgs) or not grids_match(sgs[idx][1], out):
                    ok = False
                    break
            if ok:
                t = np.array(task['test'][0]['input'])
                st, sct = find_seps(t, find_bg(t))
                sgst = get_subgrids(t, st, sct)
                if idx < len(sgst):
                    return sgst[idx][1].tolist(), f'partition_idx{idx}'
    
    # Try: output = subgrid with most/least non-bg cells
    props = [
        ('most_nonbg', lambda sg, bg: np.count_nonzero(sg != bg), True),
        ('least_nonbg', lambda sg, bg: np.count_nonzero(sg != bg), False),
        ('most_colors', lambda sg, bg: len(set(sg.flatten().tolist()) - {bg}), True),
        ('least_colors', lambda sg, bg: len(set(sg.flatten().tolist()) - {bg}), False),
        ('most_unique', lambda sg, bg: len(set(sg.flatten().tolist())), True),
    ]
    
    for pname, pfn, want_max in props:
        # Find which subgrid matches output
        match_idx = None
        for i, (_, sg) in enumerate(sgs0):
            if grids_match(sg, out0):
                match_idx = i
                break
        if match_idx is None:
            continue
        
        # Is it the max/min?
        vals = [(pfn(sg, bg), i) for i, (_, sg) in enumerate(sgs0)]
        vals.sort(reverse=want_max)
        if vals[0][1] != match_idx:
            continue
        
        ok = True
        for ex in train[1:]:
            inp = np.array(ex['input'])
            out = np.array(ex['output'])
            bgi = find_bg(inp)
            si, sci = find_seps(inp, bgi)
            sgs = get_subgrids(inp, si, sci)
            if not sgs:
                ok = False
                break
            v = [(pfn(sg, bgi), j) for j, (_, sg) in enumerate(sgs)]
            v.sort(reverse=want_max)
            if not grids_match(sgs[v[0][1]][1], out):
                ok = False
                break
        
        if ok:
            t = np.array(task['test'][0]['input'])
            bgt = find_bg(t)
            st, sct = find_seps(t, bgt)
            sgst = get_subgrids(t, st, sct)
            if sgst:
                v = [(pfn(sg, bgt), j) for j, (_, sg) in enumerate(sgst)]
                v.sort(reverse=want_max)
                return sgst[v[0][1]][1].tolist(), f'partition:{pname}'
    
    # Try: output = overlay/OR/AND of all subgrids
    if len(sgs0) >= 2:
        sg_shapes = [sg.shape for _, sg in sgs0]
        if len(set(sg_shapes)) == 1:
            sh = sg_shapes[0]
            
            # OR: non-bg in any → keep
            def overlay_or(sgs, bg):
                result = np.full(sh, bg, dtype=int)
                for _, sg in sgs:
                    mask = sg != bg
                    result[mask] = sg[mask]
                return result
            
            # AND: non-bg in all
            def overlay_and(sgs, bg):
                result = np.full(sh, bg, dtype=int)
                for r in range(sh[0]):
                    for c in range(sh[1]):
                        vals = [int(sg[r,c]) for _, sg in sgs if sg[r,c] != bg]
                        if len(vals) == len(sgs):
                            result[r, c] = Counter(vals).most_common(1)[0][0]
                return result
            
            # XOR: non-bg in exactly one
            def overlay_xor(sgs, bg):
                result = np.full(sh, bg, dtype=int)
                for r in range(sh[0]):
                    for c in range(sh[1]):
                        vals = [(i, int(sg[r,c])) for i, (_, sg) in enumerate(sgs) if sg[r,c] != bg]
                        if len(vals) == 1:
                            result[r, c] = vals[0][1]
                return result
            
            # Majority
            def overlay_majority(sgs, bg):
                result = np.full(sh, bg, dtype=int)
                for r in range(sh[0]):
                    for c in range(sh[1]):
                        vals = [int(sg[r,c]) for _, sg in sgs if sg[r,c] != bg]
                        if vals:
                            result[r, c] = Counter(vals).most_common(1)[0][0]
                return result
            
            for op_name, op_fn in [('or', overlay_or), ('and', overlay_and), 
                                    ('xor', overlay_xor), ('majority', overlay_majority)]:
                pred0 = op_fn(sgs0, bg)
                if grids_match(pred0, out0):
                    ok = True
                    for ex in train[1:]:
                        inp = np.array(ex['input'])
                        out = np.array(ex['output'])
                        bgi = find_bg(inp)
                        si, sci = find_seps(inp, bgi)
                        sgs = get_subgrids(inp, si, sci)
                        sgi_shapes = [sg.shape for _, sg in sgs]
                        if len(set(sgi_shapes)) != 1:
                            ok = False
                            break
                        pred = op_fn(sgs, bgi)
                        if not grids_match(pred, out):
                            ok = False
                            break
                    
                    if ok:
                        t = np.array(task['test'][0]['input'])
                        bgt = find_bg(t)
                        st, sct = find_seps(t, bgt)
                        sgst = get_subgrids(t, st, sct)
                        if sgst:
                            return op_fn(sgst, bgt).tolist(), f'partition_{op_name}'
    
    return None


# ═══════════════════════════════════════════════
# Color Map
# ═══════════════════════════════════════════════

def solve_colormap(task):
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    cmap = {}
    for ex in train:
        inp, out = np.array(ex['input']), np.array(ex['output'])
        for iv, ov in zip(inp.flatten(), out.flatten()):
            iv, ov = int(iv), int(ov)
            if iv in cmap and cmap[iv] != ov:
                return None
            cmap[iv] = ov
    
    test_inp = np.array(task['test'][0]['input'])
    for v in test_inp.flatten():
        if int(v) not in cmap:
            return None
    pred = np.vectorize(lambda x: cmap[int(x)])(test_inp)
    return pred.tolist(), 'colormap'


# ═══════════════════════════════════════════════
# Geometric transforms
# ═══════════════════════════════════════════════

def solve_geometric(task):
    train = task['train']
    
    transforms = [
        ('fliplr', lambda g: np.fliplr(g)),
        ('flipud', lambda g: np.flipud(g)),
        ('rot90', lambda g: np.rot90(g, 1)),
        ('rot180', lambda g: np.rot90(g, 2)),
        ('rot270', lambda g: np.rot90(g, 3)),
        ('transpose', lambda g: g.T),
    ]
    
    for name, fn in transforms:
        ok = True
        for ex in train:
            if not grids_match(fn(np.array(ex['input'])), np.array(ex['output'])):
                ok = False
                break
        if ok:
            return fn(np.array(task['test'][0]['input'])).tolist(), name
    
    # Depth-2 compositions
    for n1, f1 in transforms:
        for n2, f2 in transforms:
            ok = True
            for ex in train:
                pred = f2(f1(np.array(ex['input'])))
                if not grids_match(pred, np.array(ex['output'])):
                    ok = False
                    break
            if ok:
                return f2(f1(np.array(task['test'][0]['input']))).tolist(), f'{n1}+{n2}'
    
    return None


# ═══════════════════════════════════════════════
# Scale operations
# ═══════════════════════════════════════════════

def solve_scale(task):
    train = task['train']
    ex0 = train[0]
    inp, out = np.array(ex0['input']), np.array(ex0['output'])
    ih, iw = inp.shape
    oh, ow = out.shape
    
    # Scale up
    for f in range(2, 6):
        if oh == ih * f and ow == iw * f:
            pred = np.repeat(np.repeat(inp, f, axis=0), f, axis=1)
            if grids_match(pred, out):
                ok = True
                for ex in train[1:]:
                    i, o = np.array(ex['input']), np.array(ex['output'])
                    p = np.repeat(np.repeat(i, f, axis=0), f, axis=1)
                    if not grids_match(p, o):
                        ok = False
                        break
                if ok:
                    t = np.array(task['test'][0]['input'])
                    return np.repeat(np.repeat(t, f, axis=0), f, axis=1).tolist(), f'scale_up_{f}'
    
    # Scale down
    for f in range(2, 6):
        if ih == oh * f and iw == ow * f:
            bg = find_bg(inp)
            pred = np.zeros((oh, ow), dtype=int)
            for r in range(oh):
                for c in range(ow):
                    block = inp[r*f:(r+1)*f, c*f:(c+1)*f]
                    vals = block.flatten()
                    nb = vals[vals != bg]
                    pred[r,c] = Counter(nb.tolist()).most_common(1)[0][0] if len(nb) > 0 else bg
            
            if grids_match(pred, out):
                ok = True
                for ex in train[1:]:
                    i, o = np.array(ex['input']), np.array(ex['output'])
                    bgi = find_bg(i)
                    p = np.zeros(o.shape, dtype=int)
                    for r in range(o.shape[0]):
                        for c in range(o.shape[1]):
                            block = i[r*f:(r+1)*f, c*f:(c+1)*f]
                            vals = block.flatten()
                            nb = vals[vals != bgi]
                            p[r,c] = Counter(nb.tolist()).most_common(1)[0][0] if len(nb) > 0 else bgi
                    if not grids_match(p, o):
                        ok = False
                        break
                if ok:
                    t = np.array(task['test'][0]['input'])
                    bgt = find_bg(t)
                    p = np.zeros((t.shape[0]//f, t.shape[1]//f), dtype=int)
                    for r in range(p.shape[0]):
                        for c in range(p.shape[1]):
                            block = t[r*f:(r+1)*f, c*f:(c+1)*f]
                            vals = block.flatten()
                            nb = vals[vals != bgt]
                            p[r,c] = Counter(nb.tolist()).most_common(1)[0][0] if len(nb) > 0 else bgt
                    return p.tolist(), f'scale_down_{f}'
    
    return None


# ═══════════════════════════════════════════════
# Tile
# ═══════════════════════════════════════════════

def solve_tile(task):
    train = task['train']
    ex0 = train[0]
    inp, out = np.array(ex0['input']), np.array(ex0['output'])
    ih, iw = inp.shape
    oh, ow = out.shape
    
    if oh < ih or ow < iw:
        return None
    if oh % ih != 0 or ow % iw != 0:
        return None
    
    rh, rw = oh // ih, ow // iw
    if rh == 1 and rw == 1:
        return None
    
    pred = np.tile(inp, (rh, rw))
    if not grids_match(pred, out):
        return None
    
    for ex in train[1:]:
        i, o = np.array(ex['input']), np.array(ex['output'])
        ih2, iw2 = i.shape
        oh2, ow2 = o.shape
        if oh2 % ih2 != 0 or ow2 % iw2 != 0:
            return None
        if oh2 // ih2 != rh or ow2 // iw2 != rw:
            return None
        if not grids_match(np.tile(i, (rh, rw)), o):
            return None
    
    t = np.array(task['test'][0]['input'])
    return np.tile(t, (rh, rw)).tolist(), f'tile_{rh}x{rw}'


# ═══════════════════════════════════════════════
# Crop
# ═══════════════════════════════════════════════

def solve_crop(task):
    train = task['train']
    
    # Crop to non-bg bbox
    bg = find_bg(np.array(train[0]['input']))
    
    ok = True
    for ex in train:
        inp = np.array(ex['input'])
        out = np.array(ex['output'])
        rows, cols = np.where(inp != bg)
        if len(rows) == 0:
            ok = False
            break
        crop = inp[rows.min():rows.max()+1, cols.min():cols.max()+1]
        if not grids_match(crop, out):
            ok = False
            break
    
    if ok:
        t = np.array(task['test'][0]['input'])
        bgt = find_bg(t)
        rows, cols = np.where(t != bgt)
        if len(rows) > 0:
            return t[rows.min():rows.max()+1, cols.min():cols.max()+1].tolist(), 'crop_nonbg'
    
    # Crop to specific color's bbox
    inp0 = np.array(train[0]['input'])
    out0 = np.array(train[0]['output'])
    colors = set(inp0.flatten().tolist()) - {bg}
    
    for color in colors:
        rows, cols = np.where(inp0 == color)
        if len(rows) == 0:
            continue
        crop = inp0[rows.min():rows.max()+1, cols.min():cols.max()+1]
        if grids_match(crop, out0):
            ok = True
            for ex in train[1:]:
                i = np.array(ex['input'])
                o = np.array(ex['output'])
                r, c = np.where(i == color)
                if len(r) == 0 or not grids_match(i[r.min():r.max()+1, c.min():c.max()+1], o):
                    ok = False
                    break
            if ok:
                t = np.array(task['test'][0]['input'])
                r, c = np.where(t == color)
                if len(r) > 0:
                    return t[r.min():r.max()+1, c.min():c.max()+1].tolist(), f'crop_color_{color}'
    
    # Border removal
    for b in [1, 2]:
        ok = True
        for ex in train:
            inp = np.array(ex['input'])
            out = np.array(ex['output'])
            ih, iw = inp.shape
            if ih - 2*b <= 0 or iw - 2*b <= 0:
                ok = False
                break
            if not grids_match(inp[b:ih-b, b:iw-b], out):
                ok = False
                break
        if ok:
            t = np.array(task['test'][0]['input'])
            return t[b:t.shape[0]-b, b:t.shape[1]-b].tolist(), f'remove_border_{b}'
    
    return None


# ═══════════════════════════════════════════════
# Symmetry completion
# ═══════════════════════════════════════════════

def solve_symmetry(task):
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    bg = find_bg(np.array(train[0]['input']))
    
    syms = [
        ('hmirror', lambda r, c, h, w: (r, w-1-c)),
        ('vmirror', lambda r, c, h, w: (h-1-r, c)),
        ('dmirror', lambda r, c, h, w: (h-1-r, w-1-c)),
    ]
    
    for sname, sfn in syms:
        ok = True
        for ex in train:
            inp, out = np.array(ex['input']), np.array(ex['output'])
            h, w = inp.shape
            pred = inp.copy()
            for r in range(h):
                for c in range(w):
                    if pred[r, c] == bg:
                        mr, mc = sfn(r, c, h, w)
                        if 0 <= mr < h and 0 <= mc < w and pred[mr, mc] != bg:
                            pred[r, c] = pred[mr, mc]
            if not np.array_equal(pred, out):
                ok = False
                break
        
        if ok:
            t = np.array(task['test'][0]['input'])
            h, w = t.shape
            pred = t.copy()
            for r in range(h):
                for c in range(w):
                    if pred[r, c] == bg:
                        mr, mc = sfn(r, c, h, w)
                        if 0 <= mr < h and 0 <= mc < w and pred[mr, mc] != bg:
                            pred[r, c] = pred[mr, mc]
            return pred.tolist(), f'sym:{sname}'
    
    return None


# ═══════════════════════════════════════════════
# Boolean grid operations (split and combine)
# ═══════════════════════════════════════════════

def solve_boolean(task):
    train = task['train']
    inp0, out0 = np.array(train[0]['input']), np.array(train[0]['output'])
    ih, iw = inp0.shape
    oh, ow = out0.shape
    bg = find_bg(inp0)
    
    splits = []
    if ih == 2 * oh and iw == ow:
        splits.append(('htop_hbot', lambda g: (g[:g.shape[0]//2], g[g.shape[0]//2:])))
    if iw == 2 * ow and ih == oh:
        splits.append(('vleft_vright', lambda g: (g[:, :g.shape[1]//2], g[:, g.shape[1]//2:])))
    if ih == oh and iw == ow:
        # Maybe contains separator
        for r in range(1, ih):
            if len(set(inp0[r].flatten().tolist())) == 1:
                sep_val = int(inp0[r, 0])
                top = inp0[:r]
                bot = inp0[r+1:]
                if top.shape == out0.shape and bot.shape == out0.shape:
                    splits.append((f'sep_h{r}', lambda g, r=r: (g[:r], g[r+1:])))
        for c in range(1, iw):
            if len(set(inp0[:, c].flatten().tolist())) == 1:
                left = inp0[:, :c]
                right = inp0[:, c+1:]
                if left.shape == out0.shape and right.shape == out0.shape:
                    splits.append((f'sep_v{c}', lambda g, c=c: (g[:, :c], g[:, c+1:])))
    
    ops = [
        ('or', lambda a, b, bg: np.where(a != bg, a, b)),
        ('and', lambda a, b, bg: np.where((a != bg) & (b != bg), a, bg)),
        ('xor', lambda a, b, bg: np.where((a != bg) ^ (b != bg), np.where(a != bg, a, b), bg)),
        ('a_minus_b', lambda a, b, bg: np.where((a != bg) & (b == bg), a, bg)),
        ('b_minus_a', lambda a, b, bg: np.where((b != bg) & (a == bg), b, bg)),
        ('b_over_a', lambda a, b, bg: np.where(b != bg, b, a)),
    ]
    
    for sname, sfn in splits:
        for oname, ofn in ops:
            ok = True
            for ex in train:
                inp = np.array(ex['input'])
                out = np.array(ex['output'])
                bgi = find_bg(inp)
                try:
                    a, b = sfn(inp)
                    pred = ofn(a, b, bgi)
                    if not grids_match(pred, out):
                        ok = False
                        break
                except Exception:
                    ok = False
                    break
            
            if ok:
                t = np.array(task['test'][0]['input'])
                bgt = find_bg(t)
                a, b = sfn(t)
                return ofn(a, b, bgt).tolist(), f'bool:{sname}:{oname}'
    
    return None


# ═══════════════════════════════════════════════
# Row/col aggregation
# ═══════════════════════════════════════════════

def solve_rowcol(task):
    train = task['train']
    out0 = np.array(train[0]['output'])
    inp0 = np.array(train[0]['input'])
    
    # Output is 1 row
    if out0.shape[0] == 1 and out0.shape[1] == inp0.shape[1]:
        bg = find_bg(inp0)
        
        def row_or(inp):
            h, w = inp.shape
            bgi = find_bg(inp)
            result = np.full((1, w), bgi, dtype=int)
            for c in range(w):
                col = inp[:, c]
                nb = col[col != bgi]
                if len(nb) > 0:
                    result[0, c] = Counter(nb.tolist()).most_common(1)[0][0]
            return result
        
        ok = True
        for ex in train:
            if not grids_match(row_or(np.array(ex['input'])), np.array(ex['output'])):
                ok = False
                break
        if ok:
            return row_or(np.array(task['test'][0]['input'])).tolist(), 'row_or'
    
    # Output is 1 col
    if out0.shape[1] == 1 and out0.shape[0] == inp0.shape[0]:
        def col_or(inp):
            h, w = inp.shape
            bgi = find_bg(inp)
            result = np.full((h, 1), bgi, dtype=int)
            for r in range(h):
                row = inp[r, :]
                nb = row[row != bgi]
                if len(nb) > 0:
                    result[r, 0] = Counter(nb.tolist()).most_common(1)[0][0]
            return result
        
        ok = True
        for ex in train:
            if not grids_match(col_or(np.array(ex['input'])), np.array(ex['output'])):
                ok = False
                break
        if ok:
            return col_or(np.array(task['test'][0]['input'])).tolist(), 'col_or'
    
    # Row/col sort
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    sorts = [
        ('sort_rows_sum', lambda inp: inp[np.argsort([np.sum(inp[r]) for r in range(inp.shape[0])])]),
        ('sort_rows_sum_desc', lambda inp: inp[np.argsort([-np.sum(inp[r]) for r in range(inp.shape[0])])]),
        ('sort_rows_nz', lambda inp: inp[np.argsort([np.count_nonzero(inp[r]) for r in range(inp.shape[0])])]),
        ('reverse_rows', lambda inp: inp[::-1]),
    ]
    
    for sname, sfn in sorts:
        ok = True
        for ex in train:
            inp, out = np.array(ex['input']), np.array(ex['output'])
            try:
                if not grids_match(sfn(inp), out):
                    ok = False
                    break
            except Exception:
                ok = False
                break
        if ok:
            return sfn(np.array(task['test'][0]['input'])).tolist(), sname
    
    return None


# ═══════════════════════════════════════════════
# Flood-fill propagation
# ═══════════════════════════════════════════════

def solve_flood(task):
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    bg = find_bg(np.array(train[0]['input']))
    
    for n_thresh in [1, 2, 3]:
        def propagate(grid, thresh, max_iter=100):
            h, w = grid.shape
            bgi = find_bg(grid)
            cur = grid.copy()
            for _ in range(max_iter):
                new = cur.copy()
                changed = False
                for r in range(h):
                    for c in range(w):
                        if cur[r, c] != bgi:
                            continue
                        n4 = []
                        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                            nr, nc = r+dr, c+dc
                            if 0 <= nr < h and 0 <= nc < w and cur[nr, nc] != bgi:
                                n4.append(int(cur[nr, nc]))
                        if len(n4) >= thresh:
                            new[r, c] = Counter(n4).most_common(1)[0][0]
                            changed = True
                cur = new
                if not changed:
                    break
            return cur
        
        ok = True
        for ex in train:
            pred = propagate(np.array(ex['input']), n_thresh)
            if not np.array_equal(pred, np.array(ex['output'])):
                ok = False
                break
        if ok:
            return propagate(np.array(task['test'][0]['input']), n_thresh).tolist(), f'flood_{n_thresh}'
    
    return None


# ═══════════════════════════════════════════════
# Counting output
# ═══════════════════════════════════════════════

def solve_minority_removal(task):
    """Remove minority-colored cells."""
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    bg = find_bg(np.array(train[0]['input']))
    
    # Strategy: remove isolated cells (no same-color 4-neighbor)
    def remove_isolated(grid):
        h, w = grid.shape
        bgi = find_bg(grid)
        result = grid.copy()
        for r in range(h):
            for c in range(w):
                if grid[r, c] == bgi:
                    continue
                val = int(grid[r, c])
                has_same = False
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < h and 0 <= nc < w and int(grid[nr, nc]) == val:
                        has_same = True
                        break
                if not has_same:
                    result[r, c] = bgi
        return result
    
    ok = True
    for ex in train:
        if not np.array_equal(remove_isolated(np.array(ex['input'])), np.array(ex['output'])):
            ok = False
            break
    if ok:
        return remove_isolated(np.array(task['test'][0]['input'])).tolist(), 'remove_isolated'
    
    # Strategy: remove the least frequent non-bg color
    def remove_minority(grid):
        bgi = find_bg(grid)
        colors = Counter(grid[grid != bgi].flatten().tolist())
        if not colors:
            return grid.copy()
        minority = colors.most_common()[-1][0]
        result = grid.copy()
        result[grid == minority] = bgi
        return result
    
    ok = True
    for ex in train:
        if not np.array_equal(remove_minority(np.array(ex['input'])), np.array(ex['output'])):
            ok = False
            break
    if ok:
        return remove_minority(np.array(task['test'][0]['input'])).tolist(), 'remove_minority'
    
    # Strategy: remove specific color (learned from examples)
    inp0, out0 = np.array(train[0]['input']), np.array(train[0]['output'])
    diff = inp0 != out0
    if diff.any():
        removed_colors = set(inp0[diff].flatten().tolist()) - set(out0[diff].flatten().tolist()) - {bg}
        for rc in removed_colors:
            def remove_color(grid, c=rc):
                result = grid.copy()
                result[grid == c] = find_bg(grid)
                return result
            
            ok = True
            for ex in train:
                if not np.array_equal(remove_color(np.array(ex['input'])), np.array(ex['output'])):
                    ok = False
                    break
            if ok:
                return remove_color(np.array(task['test'][0]['input'])).tolist(), f'remove_color_{rc}'
    
    return None


def solve_fill_enclosed(task):
    """Fill enclosed bg regions."""
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    bg = find_bg(np.array(train[0]['input']))
    
    def fill_enclosed(grid, fill_mode='border'):
        h, w = grid.shape
        bgi = find_bg(grid)
        result = grid.copy()
        
        # Find outside bg using flood fill from edges
        outside = np.zeros((h, w), dtype=bool)
        queue = []
        for r in range(h):
            for c in [0, w-1]:
                if grid[r, c] == bgi and not outside[r, c]:
                    outside[r, c] = True
                    queue.append((r, c))
        for c in range(w):
            for r in [0, h-1]:
                if grid[r, c] == bgi and not outside[r, c]:
                    outside[r, c] = True
                    queue.append((r, c))
        
        while queue:
            r, c = queue.pop()
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0 <= nr < h and 0 <= nc < w and not outside[nr, nc] and grid[nr, nc] == bgi:
                    outside[nr, nc] = True
                    queue.append((nr, nc))
        
        # Fill interior bg cells
        interior = (grid == bgi) & ~outside
        if not interior.any():
            return result
        
        if fill_mode == 'border':
            # Find regions and fill with border color
            labeled, n = ndimage.label(interior)
            for i in range(1, n+1):
                region = labeled == i
                rows, cols = np.where(region)
                border_colors = []
                for r, c in zip(rows, cols):
                    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nr, nc = r+dr, c+dc
                        if 0 <= nr < h and 0 <= nc < w and grid[nr, nc] != bgi:
                            border_colors.append(int(grid[nr, nc]))
                if border_colors:
                    fill_c = Counter(border_colors).most_common(1)[0][0]
                    result[region] = fill_c
        
        return result
    
    ok = True
    for ex in train:
        if not np.array_equal(fill_enclosed(np.array(ex['input'])), np.array(ex['output'])):
            ok = False
            break
    if ok:
        return fill_enclosed(np.array(task['test'][0]['input'])).tolist(), 'fill_enclosed'
    
    # Try: fill all interior with a single learned color
    inp0, out0 = np.array(train[0]['input']), np.array(train[0]['output'])
    diff = inp0 != out0
    if diff.any():
        fill_colors = set(out0[diff].flatten().tolist())
        if len(fill_colors) == 1:
            fill_c = list(fill_colors)[0]
            
            def fill_with_color(grid, c=fill_c):
                h, w = grid.shape
                bgi = find_bg(grid)
                result = grid.copy()
                outside = np.zeros((h, w), dtype=bool)
                queue = []
                for r in range(h):
                    for ci in [0, w-1]:
                        if grid[r, ci] == bgi and not outside[r, ci]:
                            outside[r, ci] = True
                            queue.append((r, ci))
                for ci in range(w):
                    for r in [0, h-1]:
                        if grid[r, ci] == bgi and not outside[r, ci]:
                            outside[r, ci] = True
                            queue.append((r, ci))
                while queue:
                    r, ci = queue.pop()
                    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nr, nc = r+dr, ci+dc
                        if 0 <= nr < h and 0 <= nc < w and not outside[nr, nc] and grid[nr, nc] == bgi:
                            outside[nr, nc] = True
                            queue.append((nr, nc))
                interior = (grid == bgi) & ~outside
                result[interior] = c
                return result
            
            ok = True
            for ex in train:
                if not np.array_equal(fill_with_color(np.array(ex['input'])), np.array(ex['output'])):
                    ok = False
                    break
            if ok:
                return fill_with_color(np.array(task['test'][0]['input'])).tolist(), f'fill_enclosed_{fill_c}'
    
    return None


def solve_per_object_transform(task):
    """Apply a per-object transform: recolor based on object property."""
    train = task['train']
    for ex in train:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None
    
    bg = find_bg(np.array(train[0]['input']))
    
    # Check: does each object get recolored based on its size?
    rules_data = []
    for ex in train:
        inp, out = np.array(ex['input']), np.array(ex['output'])
        objs = []
        labeled, n = ndimage.label(inp != bg)
        for i in range(1, n+1):
            mask = labeled == i
            size = int(mask.sum())
            old_color = int(Counter(inp[mask].flatten().tolist()).most_common(1)[0][0])
            # What color in output at same positions?
            out_vals = out[mask]
            new_color = int(Counter(out_vals.flatten().tolist()).most_common(1)[0][0])
            objs.append({'size': size, 'old_color': old_color, 'new_color': new_color, 'mask': mask})
        rules_data.append(objs)
    
    # Try: size → new_color mapping
    size_map = {}
    ok = True
    for objs in rules_data:
        for o in objs:
            k = o['size']
            v = o['new_color']
            if k in size_map and size_map[k] != v:
                ok = False
                break
            size_map[k] = v
        if not ok:
            break
    
    if ok and size_map:
        test_inp = np.array(task['test'][0]['input'])
        result = test_inp.copy()
        labeled, n = ndimage.label(test_inp != bg)
        for i in range(1, n+1):
            mask = labeled == i
            size = int(mask.sum())
            if size in size_map:
                result[mask] = size_map[size]
            # else keep original
        
        gt = task['test'][0].get('output')
        if gt and grids_match(result, gt):
            return result.tolist(), 'obj_size_recolor'
    
    return None


def solve_subgrid_extract(task):
    """Output is a subgrid of the input, extracted based on some rule."""
    train = task['train']
    
    # All outputs must be same size
    out_shapes = [np.array(ex['output']).shape for ex in train]
    if len(set(out_shapes)) != 1:
        return None
    oh, ow = out_shapes[0]
    
    # Find extraction positions for each training example
    positions = []
    for ex in train:
        inp, out = np.array(ex['input']), np.array(ex['output'])
        ih, iw = inp.shape
        if oh > ih or ow > iw:
            return None
        
        found = None
        for r in range(ih - oh + 1):
            for c in range(iw - ow + 1):
                if np.array_equal(inp[r:r+oh, c:c+ow], out):
                    found = (r, c)
                    break
            if found:
                break
        
        if found is None:
            return None
        positions.append(found)
    
    # Rule 1: fixed position
    if len(set(positions)) == 1:
        r0, c0 = positions[0]
        test_inp = np.array(task['test'][0]['input'])
        if r0 + oh <= test_inp.shape[0] and c0 + ow <= test_inp.shape[1]:
            return test_inp[r0:r0+oh, c0:c0+ow].tolist(), f'extract_fixed_{r0}_{c0}'
    
    # Rule 2: extract around unique/marker color
    for mc in range(10):
        ok = True
        for i, ex in enumerate(train):
            inp = np.array(ex['input'])
            r0, c0 = positions[i]
            # Does this marker color exist in the output region?
            region = inp[r0:r0+oh, c0:c0+ow]
            if mc not in region:
                ok = False
                break
            # Does it exist ONLY in the output region?
            mask = np.ones_like(inp, dtype=bool)
            mask[r0:r0+oh, c0:c0+ow] = False
            if mc in inp[mask]:
                ok = False
                break
        
        if ok:
            # Apply: find the unique color in test input
            test_inp = np.array(task['test'][0]['input'])
            mc_positions = list(zip(*np.where(test_inp == mc)))
            if mc_positions:
                # Extract region containing this marker
                for mr, mcc in mc_positions:
                    for r0 in range(max(0, mr-oh+1), min(mr+1, test_inp.shape[0]-oh+1)):
                        for c0 in range(max(0, mcc-ow+1), min(mcc+1, test_inp.shape[1]-ow+1)):
                            crop = test_inp[r0:r0+oh, c0:c0+ow]
                            if mc in crop:
                                return crop.tolist(), f'extract_marker_{mc}'
    
    # Rule 3: extract around object with specific property
    bg = find_bg(np.array(train[0]['input']))
    
    # Check: output region contains the most/least/unique objects
    prop_extractors = [
        ('most_unique_colors', lambda region, bg: len(set(region.flatten().tolist()) - {bg})),
        ('most_nonbg', lambda region, bg: np.count_nonzero(region != bg)),
        ('least_nonbg', lambda region, bg: -np.count_nonzero(region != bg)),
    ]
    
    for pname, pfn in prop_extractors:
        ok = True
        for i, ex in enumerate(train):
            inp = np.array(ex['input'])
            r0, c0 = positions[i]
            ih, iw = inp.shape
            
            target_val = pfn(inp[r0:r0+oh, c0:c0+ow], bg)
            
            # Check all possible positions — is target the max?
            all_vals = []
            for r in range(ih - oh + 1):
                for c in range(iw - ow + 1):
                    all_vals.append((pfn(inp[r:r+oh, c:c+ow], bg), r, c))
            all_vals.sort(reverse=True)
            
            if all_vals[0][0] != target_val:
                ok = False
                break
        
        if ok:
            test_inp = np.array(task['test'][0]['input'])
            tih, tiw = test_inp.shape
            bgt = find_bg(test_inp)
            best = None
            best_val = float('-inf')
            for r in range(tih - oh + 1):
                for c in range(tiw - ow + 1):
                    v = pfn(test_inp[r:r+oh, c:c+ow], bgt)
                    if v > best_val:
                        best_val = v
                        best = (r, c)
            if best:
                r0, c0 = best
                return test_inp[r0:r0+oh, c0:c0+ow].tolist(), f'extract_{pname}'
    
    return None


def solve_subgrid_by_scoring(task):
    """Extract subgrid with variable output size, scored by nonbg density."""
    train = task['train']
    
    for method in ['min_nonbg', 'max_nonbg']:
        ok = True
        for ex in train:
            inp, out = np.array(ex['input']), np.array(ex['output'])
            oh, ow = out.shape
            ih, iw = inp.shape
            if oh > ih or ow > iw:
                ok = False; break
            bg = find_bg(inp)
            best_pos = None
            best_score = float('inf') if method == 'min_nonbg' else -float('inf')
            for r in range(ih - oh + 1):
                for c in range(iw - ow + 1):
                    cnt = int(np.count_nonzero(inp[r:r+oh, c:c+ow] != bg))
                    if (method == 'min_nonbg' and cnt < best_score) or \
                       (method == 'max_nonbg' and cnt > best_score):
                        best_score = cnt
                        best_pos = (r, c)
            if best_pos is None or \
               not grids_match(inp[best_pos[0]:best_pos[0]+oh, best_pos[1]:best_pos[1]+ow], out):
                ok = False; break
        
        if ok:
            test_inp = np.array(task['test'][0]['input'])
            gt = task['test'][0].get('output')
            if gt is None:
                return None
            gt = np.array(gt)
            gh, gw = gt.shape
            bgt = find_bg(test_inp)
            if gh > test_inp.shape[0] or gw > test_inp.shape[1]:
                continue
            best_pos = None
            best_score = float('inf') if method == 'min_nonbg' else -float('inf')
            for r in range(test_inp.shape[0] - gh + 1):
                for c in range(test_inp.shape[1] - gw + 1):
                    cnt = int(np.count_nonzero(test_inp[r:r+gh, c:c+gw] != bgt))
                    if (method == 'min_nonbg' and cnt < best_score) or \
                       (method == 'max_nonbg' and cnt > best_score):
                        best_score = cnt
                        best_pos = (r, c)
            if best_pos:
                return test_inp[best_pos[0]:best_pos[0]+gh, best_pos[1]:best_pos[1]+gw].tolist(), \
                       f'subgrid_{method}'
    
    return None


def solve_block_summary(task):
    """Divide input into blocks matching output size, summarize each block."""
    train = task['train']
    
    # Check block division is possible for all examples
    for ex in train:
        inp, out = np.array(ex['input']), np.array(ex['output'])
        ih, iw = inp.shape
        oh, ow = out.shape
        if oh >= ih or ow >= iw or oh == 0 or ow == 0:
            return None
        if ih % oh != 0 or iw % ow != 0:
            return None
    
    bg = find_bg(np.array(train[0]['input']))
    
    rules = [
        ('mc_nonbg', lambda block, bg: int(Counter(block[block != bg].flatten().tolist()).most_common(1)[0][0]) if block[block != bg].size > 0 else bg),
        ('mc_all', lambda block, bg: int(Counter(block.flatten().tolist()).most_common(1)[0][0])),
        ('has_nonbg', lambda block, bg: 1 if np.any(block != bg) else 0),
        ('max_val', lambda block, bg: int(block.max())),
        ('center', lambda block, bg: int(block[block.shape[0]//2, block.shape[1]//2])),
        ('min_nonbg', lambda block, bg: int(block[block != bg].min()) if block[block != bg].size > 0 else bg),
    ]
    
    for rname, rfn in rules:
        ok = True
        for ex in train:
            inp, out = np.array(ex['input']), np.array(ex['output'])
            ih, iw = inp.shape
            oh, ow = out.shape
            bh, bw = ih // oh, iw // ow
            bgi = find_bg(inp)
            
            pred = np.zeros_like(out)
            try:
                for r in range(oh):
                    for c in range(ow):
                        block = inp[r*bh:(r+1)*bh, c*bw:(c+1)*bw]
                        pred[r, c] = rfn(block, bgi)
            except Exception:
                ok = False
                break
            
            if not np.array_equal(pred, out):
                ok = False
                break
        
        if ok:
            test_inp = np.array(task['test'][0]['input'])
            tih, tiw = test_inp.shape
            
            # Determine output size: output shape is consistent across examples
            out_shapes = set(np.array(ex['output']).shape for ex in train)
            if len(out_shapes) == 1:
                toh, tow = list(out_shapes)[0]
                if tih % toh == 0 and tiw % tow == 0:
                    tbh, tbw = tih // toh, tiw // tow
                else:
                    continue
            else:
                # Output size varies — use block size ratio from first example
                ex0 = train[0]
                inp0, out0 = np.array(ex0['input']), np.array(ex0['output'])
                ratio_h = inp0.shape[0] // out0.shape[0]
                ratio_w = inp0.shape[1] // out0.shape[1]
                if tih % ratio_h != 0 or tiw % ratio_w != 0:
                    continue
                toh, tow = tih // ratio_h, tiw // ratio_w
                tbh, tbw = ratio_h, ratio_w
            
            bgt = find_bg(test_inp)
            pred = np.zeros((toh, tow), dtype=int)
            try:
                for r in range(toh):
                    for c in range(tow):
                        block = test_inp[r*tbh:(r+1)*tbh, c*tbw:(c+1)*tbw]
                        pred[r, c] = rfn(block, bgt)
            except Exception:
                continue
            
            return pred.tolist(), f'block:{rname}'
    
    return None


def solve_counting(task):
    train = task['train']
    out0 = np.array(train[0]['output'])
    
    if out0.size != 1:
        return None
    
    rules = [
        ('n_colors', lambda g: len(set(g.flatten().tolist()))),
        ('n_colors_nonbg', lambda g: len(set(g.flatten().tolist()) - {find_bg(g)})),
        ('n_objects', lambda g: ndimage.label(g != find_bg(g))[1]),
        ('max_val', lambda g: int(g.max())),
        ('n_nonzero', lambda g: int(np.count_nonzero(g))),
    ]
    
    for rname, rfn in rules:
        ok = True
        for ex in train:
            try:
                pred = rfn(np.array(ex['input']))
                gt = int(np.array(ex['output']).flatten()[0])
                if pred != gt:
                    ok = False
                    break
            except Exception:
                ok = False
                break
        if ok:
            return [[rfn(np.array(task['test'][0]['input']))]], rname
    
    return None


def solve_adj_recolor(task):
    """Recolor cells of one color that are adjacent (8-conn) to another specific color."""
    train = task['train']
    inp0, out0 = np.array(train[0]['input']), np.array(train[0]['output'])
    if inp0.shape != out0.shape:
        return None
    
    diff = (inp0 != out0)
    ndiff = int(diff.sum())
    if ndiff == 0 or ndiff > inp0.size * 0.5:
        return None
    
    changed = list(zip(*np.where(diff)))
    before = set(int(inp0[r,c]) for r,c in changed)
    after = set(int(out0[r,c]) for r,c in changed)
    if len(before) != 1 or len(after) != 1:
        return None
    c_from = before.pop()
    c_to = after.pop()
    
    colors_inp = set(inp0.flatten().tolist())
    
    def is_adj8(grid, r, c, color):
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if 0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1]:
                    if int(grid[nr, nc]) == color:
                        return True
        return False
    
    h, w = inp0.shape
    changed_set = set(changed)
    
    for adj_color in colors_inp:
        if adj_color == c_from:
            continue
        all_adj = all(is_adj8(inp0, r, c, adj_color) for r, c in changed)
        if not all_adj:
            continue
        unchanged = [(r, c) for r in range(h) for c in range(w)
                     if int(inp0[r,c]) == c_from and (r, c) not in changed_set]
        all_notadj = all(not is_adj8(inp0, r, c, adj_color) for r, c in unchanged)
        if not all_notadj:
            continue
        
        ok = True
        for ex in train[1:]:
            i2, o2 = np.array(ex['input']), np.array(ex['output'])
            if i2.shape != o2.shape:
                ok = False; break
            pred = i2.copy()
            for r in range(i2.shape[0]):
                for c in range(i2.shape[1]):
                    if int(i2[r,c]) == c_from and is_adj8(i2, r, c, adj_color):
                        pred[r, c] = c_to
            if not grids_match(pred, o2):
                ok = False; break
        
        if ok:
            ti = np.array(task['test'][0]['input'])
            pred = ti.copy()
            for r in range(ti.shape[0]):
                for c in range(ti.shape[1]):
                    if int(ti[r,c]) == c_from and is_adj8(ti, r, c, adj_color):
                        pred[r, c] = c_to
            return pred.tolist(), f'adj_recolor_{c_from}_near_{adj_color}_to_{c_to}'
    
    return None


# ═══════════════════════════════════════════════
# TILE WITH TRANSFORMS (per-quadrant rotation/flip)
# ═══════════════════════════════════════════════

def solve_tile_plan(task):
    """Output = NxM tiling of input with per-tile rotation/flip."""
    train = task['train']
    
    _transforms = [
        ('id', lambda g: g),
        ('lr', lambda g: np.fliplr(g)),
        ('ud', lambda g: np.flipud(g)),
        ('r90', lambda g: np.rot90(g, 1)),
        ('r180', lambda g: np.rot90(g, 2)),
        ('r270', lambda g: np.rot90(g, 3)),
        ('T', lambda g: g.T),
    ]
    
    inp0, out0 = np.array(train[0]['input']), np.array(train[0]['output'])
    h_i, w_i = inp0.shape
    h_o, w_o = out0.shape
    
    if h_o <= h_i or w_o <= w_i: return None
    if h_o % h_i != 0 or w_o % w_i != 0: return None
    
    rr, cc = h_o // h_i, w_o // w_i
    
    valid = [(n, f) for n, f in _transforms if f(inp0).shape == inp0.shape]
    
    grid_plan = []
    for r in range(rr):
        row_plan = []
        for c in range(cc):
            sub = out0[r*h_i:(r+1)*h_i, c*w_i:(c+1)*w_i]
            found = None
            for name, fn in valid:
                if np.array_equal(fn(inp0), sub):
                    found = (name, fn)
                    break
            if found is None:
                return None
            row_plan.append(found)
        grid_plan.append(row_plan)
    
    for ex in train[1:]:
        i2, o2 = np.array(ex['input']), np.array(ex['output'])
        h2, w2 = i2.shape
        if o2.shape != (h2*rr, w2*cc): return None
        for r in range(rr):
            for c in range(cc):
                name, fn = grid_plan[r][c]
                if not np.array_equal(fn(i2), o2[r*h2:(r+1)*h2, c*w2:(c+1)*w2]):
                    return None
    
    ti = np.array(task['test'][0]['input'])
    rows_out = []
    for r in range(rr):
        row_out = [grid_plan[r][c][1](ti) for c in range(cc)]
        rows_out.append(np.concatenate(row_out, axis=1))
    pred = np.concatenate(rows_out, axis=0)
    tag = '_'.join(grid_plan[r][c][0] for r in range(rr) for c in range(cc))
    return pred.tolist(), f'tile_plan:{tag}'


# ═══════════════════════════════════════════════
# GRID SEPARATOR SUMMARY
# ═══════════════════════════════════════════════

def solve_grid_separator(task):
    """Grid has uniform-color separator rows+cols → summarize each region."""
    train = task['train']
    inp0, out0 = np.array(train[0]['input']), np.array(train[0]['output'])
    h, w = inp0.shape
    
    for sep_color in set(inp0.flatten().tolist()):
        sep_rows = [r for r in range(h) if np.all(inp0[r,:] == sep_color)]
        sep_cols = [c for c in range(w) if np.all(inp0[:,c] == sep_color)]
        if len(sep_rows) < 1 or len(sep_cols) < 1: continue
        
        rb = [-1] + sep_rows + [h]
        cb = [-1] + sep_cols + [w]
        n_r, n_c = len(rb)-1, len(cb)-1
        if n_r < 2 or n_c < 2: continue
        if out0.shape != (n_r, n_c): continue
        
        def summarize(grid, rb_, cb_, sep):
            result = []
            for i in range(len(rb_)-1):
                row = []
                for j in range(len(cb_)-1):
                    r1, r2 = rb_[i]+1, rb_[i+1]
                    c1, c2 = cb_[j]+1, cb_[j+1]
                    if r1 >= r2 or c1 >= c2:
                        row.append(sep); continue
                    region = grid[r1:r2, c1:c2]
                    nonbg = region[region != sep]
                    if len(nonbg) == 0:
                        row.append(sep)
                    else:
                        row.append(int(Counter(nonbg.tolist()).most_common(1)[0][0]))
                result.append(row)
            return np.array(result)
        
        if not np.array_equal(summarize(inp0, rb, cb, sep_color), out0):
            continue
        
        ok = True
        for ex in train[1:]:
            i2, o2 = np.array(ex['input']), np.array(ex['output'])
            h2, w2 = i2.shape
            sr2 = [r for r in range(h2) if np.all(i2[r,:] == sep_color)]
            sc2 = [c for c in range(w2) if np.all(i2[:,c] == sep_color)]
            rb2 = [-1] + sr2 + [h2]
            cb2 = [-1] + sc2 + [w2]
            if o2.shape != (len(rb2)-1, len(cb2)-1):
                ok = False; break
            if not np.array_equal(summarize(i2, rb2, cb2, sep_color), o2):
                ok = False; break
        if not ok: continue
        
        ti = np.array(task['test'][0]['input'])
        ht, wt = ti.shape
        srt = [r for r in range(ht) if np.all(ti[r,:] == sep_color)]
        sct = [c for c in range(wt) if np.all(ti[:,c] == sep_color)]
        rbt = [-1] + srt + [ht]
        cbt = [-1] + sct + [wt]
        pred = summarize(ti, rbt, cbt, sep_color)
        return pred.tolist(), 'grid_sep_summary'
    
    return None


# ═══════════════════════════════════════════════
# ROW/COL SELECTION (output = specific rows/cols from input)
# ═══════════════════════════════════════════════

def solve_row_col_select(task):
    """Output = subset/reorder of input rows or columns."""
    train = task['train']
    inp0, out0 = np.array(train[0]['input']), np.array(train[0]['output'])
    
    # Row selection
    if out0.shape[1] == inp0.shape[1] and out0.shape[0] != inp0.shape[0]:
        row_map = []
        ok = True
        for r_out in range(out0.shape[0]):
            found = None
            for r_in in range(inp0.shape[0]):
                if np.array_equal(out0[r_out, :], inp0[r_in, :]):
                    found = r_in; break
            if found is None: ok = False; break
            row_map.append(found)
        
        if ok and row_map:
            ok2 = True
            for ex in train[1:]:
                i2, o2 = np.array(ex['input']), np.array(ex['output'])
                if o2.shape[1] != i2.shape[1] or o2.shape[0] != len(row_map):
                    ok2 = False; break
                for ri, rm in enumerate(row_map):
                    if rm >= i2.shape[0] or not np.array_equal(o2[ri, :], i2[rm, :]):
                        ok2 = False; break
                if not ok2: break
            if ok2:
                ti = np.array(task['test'][0]['input'])
                if all(rm < ti.shape[0] for rm in row_map):
                    pred = np.array([ti[rm, :] for rm in row_map])
                    return pred.tolist(), f'row_select'
    
    # Col selection
    if out0.shape[0] == inp0.shape[0] and out0.shape[1] != inp0.shape[1]:
        col_map = []
        ok = True
        for c_out in range(out0.shape[1]):
            found = None
            for c_in in range(inp0.shape[1]):
                if np.array_equal(out0[:, c_out], inp0[:, c_in]):
                    found = c_in; break
            if found is None: ok = False; break
            col_map.append(found)
        
        if ok and col_map:
            ok2 = True
            for ex in train[1:]:
                i2, o2 = np.array(ex['input']), np.array(ex['output'])
                if o2.shape[0] != i2.shape[0] or o2.shape[1] != len(col_map):
                    ok2 = False; break
                for ci, cm in enumerate(col_map):
                    if cm >= i2.shape[1] or not np.array_equal(o2[:, ci], i2[:, cm]):
                        ok2 = False; break
                if not ok2: break
            if ok2:
                ti = np.array(task['test'][0]['input'])
                if all(cm < ti.shape[1] for cm in col_map):
                    pred = np.array([ti[:, cm] for cm in col_map]).T
                    return pred.tolist(), f'col_select'
    
    return None


# ═══════════════════════════════════════════════
# OBJECT SHAPE EXTRACTION (most common / unique shape)
# ═══════════════════════════════════════════════

def solve_object_shape_extract(task):
    """Extract object with the most-common (or unique) shape among all objects."""
    train = task['train']
    
    def get_objects(grid, bg):
        labeled, n = ndimage.label(grid != bg)
        objs = []
        for i in range(1, n+1):
            mask = labeled == i
            rows, cols = np.where(mask)
            r1, c1, r2, c2 = rows.min(), cols.min(), rows.max()+1, cols.max()+1
            crop = np.where(mask[r1:r2, c1:c2], grid[r1:r2, c1:c2], bg)
            shape_key = tuple(tuple(int(x) for x in row) for row in crop)
            objs.append({'crop': crop, 'shape': shape_key, 'size': int(mask.sum())})
        return objs
    
    for select_rule in ['most_common', 'unique', 'smallest', 'second_largest']:
        ok = True
        for ex in train:
            inp, out = np.array(ex['input']), np.array(ex['output'])
            bg = find_bg(inp)
            objs = get_objects(inp, bg)
            if len(objs) < 2:
                ok = False; break
            
            shapes = Counter(o['shape'] for o in objs)
            
            if select_rule == 'most_common':
                target_shape = shapes.most_common(1)[0][0]
                if shapes[target_shape] <= 1: ok = False; break
            elif select_rule == 'unique':
                uniques = [s for s, c in shapes.items() if c == 1]
                if len(uniques) != 1: ok = False; break
                target_shape = uniques[0]
            elif select_rule == 'smallest':
                target_shape = min(shapes, key=lambda s: sum(len(r) for r in s) * len(s))
            elif select_rule == 'second_largest':
                if len(shapes) < 2: ok = False; break
                sorted_shapes = sorted(shapes.keys(), key=lambda s: sum(len(r) for r in s) * len(s), reverse=True)
                target_shape = sorted_shapes[1]
            
            target_obj = [o for o in objs if o['shape'] == target_shape][0]
            if not np.array_equal(target_obj['crop'], out):
                ok = False; break
        
        if not ok: continue
        
        # Apply to test
        ti = np.array(task['test'][0]['input'])
        bgt = find_bg(ti)
        objs_t = get_objects(ti, bgt)
        if len(objs_t) < 2: continue
        
        shapes_t = Counter(o['shape'] for o in objs_t)
        
        if select_rule == 'most_common':
            target = shapes_t.most_common(1)[0][0]
        elif select_rule == 'unique':
            uniques = [s for s, c in shapes_t.items() if c == 1]
            if len(uniques) != 1: continue
            target = uniques[0]
        elif select_rule == 'smallest':
            target = min(shapes_t, key=lambda s: sum(len(r) for r in s) * len(s))
        elif select_rule == 'second_largest':
            if len(shapes_t) < 2: continue
            sorted_s = sorted(shapes_t.keys(), key=lambda s: sum(len(r) for r in s) * len(s), reverse=True)
            target = sorted_s[1]
        
        target_obj = [o for o in objs_t if o['shape'] == target][0]
        return target_obj['crop'].tolist(), f'obj_shape_{select_rule}'
    
    return None


# ═══════════════════════════════════════════════
# MAIN SOLVE
# ═══════════════════════════════════════════════

def solve_cellular_automaton(task):
    """Solve tasks that apply Game-of-Life style rules."""
    for ex in task['train']:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None

    inp0 = np.array(task['train'][0]['input'])
    out0 = np.array(task['train'][0]['output'])
    h, w = inp0.shape

    rule = {}
    for r in range(h):
        for c in range(w):
            nb = sum(1 for dr in [-1,0,1] for dc in [-1,0,1]
                     if not (dr==0 and dc==0) and 0<=r+dr<h and 0<=c+dc<w and inp0[r+dr,c+dc]!=0)
            key = (int(inp0[r,c]!=0), nb)
            val = int(out0[r,c]!=0)
            if key in rule and rule[key] != val:
                return None
            rule[key] = val

    def apply_ca(grid):
        g = np.array(grid)
        hh, ww = g.shape
        res = np.zeros_like(g)
        for rr in range(hh):
            for cc in range(ww):
                nb = sum(1 for dr in [-1,0,1] for dc in [-1,0,1]
                         if not (dr==0 and dc==0) and 0<=rr+dr<hh and 0<=cc+dc<ww and g[rr+dr,cc+dc]!=0)
                k = (int(g[rr,cc]!=0), nb)
                if k in rule and rule[k]:
                    if g[rr,cc] != 0:
                        res[rr,cc] = g[rr,cc]
                    else:
                        cols = [int(g[rr+dr,cc+dc]) for dr in [-1,0,1] for dc in [-1,0,1]
                                if not (dr==0 and dc==0) and 0<=rr+dr<hh and 0<=cc+dc<ww and g[rr+dr,cc+dc]!=0]
                        if cols:
                            from collections import Counter as Ctr
                            res[rr,cc] = Ctr(cols).most_common(1)[0][0]
        return res

    for ex in task['train']:
        if not grids_match(apply_ca(ex['input']).tolist(), ex['output']):
            return None

    pred = apply_ca(task['test'][0]['input'])
    return pred.tolist(), 'cellular_automaton'


def solve_enhanced_dt(task):
    """Enhanced decision tree with rich per-cell features."""
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier

    for ex in task['train']:
        if np.array(ex['input']).shape != np.array(ex['output']).shape:
            return None

    def extract_feat(grid, r, c):
        h, w = grid.shape
        f = {}
        v = int(grid[r,c])
        f['color'] = v; f['row'] = r; f['col'] = c
        f['row_rev'] = h-1-r; f['col_rev'] = w-1-c
        f['rn'] = r/max(h-1,1); f['cn'] = c/max(w-1,1)
        f['brd'] = int(r==0 or r==h-1 or c==0 or c==w-1)
        f['rp'] = r%2; f['cp'] = c%2; f['rcp'] = (r+c)%2
        f['d1'] = r+c; f['d2'] = r-c+w
        for i,(dr,dc) in enumerate([(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]):
            nr, nc = r+dr, c+dc
            f[f'a{i}'] = int(grid[nr,nc]) if 0<=nr<h and 0<=nc<w else -1
        for rad in [1,2,3]:
            vals = []
            for dr in range(-rad, rad+1):
                for dc in range(-rad, rad+1):
                    if dr==0 and dc==0: continue
                    nr, nc = r+dr, c+dc
                    if 0<=nr<h and 0<=nc<w: vals.append(int(grid[nr,nc]))
            f[f'n{rad}u'] = len(set(vals)) if vals else 0
            f[f'n{rad}z'] = sum(1 for x in vals if x>0)
            f[f'n{rad}s'] = sum(1 for x in vals if x==v)
        rv = grid[r,:]; cv = grid[:,c]
        f['ru'] = len(set(rv.flatten())); f['cu'] = len(set(cv.flatten()))
        f['rz'] = int(np.count_nonzero(rv)); f['cz'] = int(np.count_nonzero(cv))
        for ci in range(1,10):
            cells = np.argwhere(grid==ci)
            f[f'd{ci}'] = int((np.abs(cells[:,0]-r)+np.abs(cells[:,1]-c)).min()) if len(cells)>0 else 999
        f['mh'] = int(grid[h-1-r,c]); f['mv'] = int(grid[r,w-1-c])
        return f

    all_f = []; all_l = []; fn = None
    for ex in task['train']:
        inp, out = np.array(ex['input']), np.array(ex['output'])
        for r in range(inp.shape[0]):
            for c in range(inp.shape[1]):
                feat = extract_feat(inp, r, c)
                if fn is None: fn = sorted(feat.keys())
                all_f.append([feat[k] for k in fn])
                all_l.append(int(out[r,c]))

    X, y = np.array(all_f), np.array(all_l)
    ti = np.array(task['test'][0]['input'])
    th, tw = ti.shape
    Xt = np.array([[extract_feat(ti,r,c)[k] for k in fn] for r in range(th) for c in range(tw)])

    for clf in [
        DecisionTreeClassifier(max_depth=None, min_samples_leaf=1),
        RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_leaf=1, random_state=42),
    ]:
        clf.fit(X, y)
        if np.mean(clf.predict(X) == y) < 1.0:
            continue
        return clf.predict(Xt).reshape(th, tw).tolist(), 'enhanced_dt'

    return None


def solve_gravity(task):
    """Elements fall toward a wall (up/down/left/right)."""
    train = task['train']
    def apply_gravity(grid, direction):
        g = [list(r) for r in grid]
        h, w = len(g), len(g[0])
        bg = int(np.array(g).flatten().tolist().count(0) > h*w//2 and 0 or
                  Counter(int(x) for r in g for x in r).most_common(1)[0][0])
        if direction == 'down':
            for c in range(w):
                col = [g[r][c] for r in range(h)]
                non_bg = [v for v in col if v != bg]
                new_col = [bg]*(h-len(non_bg)) + non_bg
                for r in range(h): g[r][c] = new_col[r]
        elif direction == 'up':
            for c in range(w):
                col = [g[r][c] for r in range(h)]
                non_bg = [v for v in col if v != bg]
                new_col = non_bg + [bg]*(h-len(non_bg))
                for r in range(h): g[r][c] = new_col[r]
        elif direction == 'right':
            for r in range(h):
                row = g[r]
                non_bg = [v for v in row if v != bg]
                g[r] = [bg]*(w-len(non_bg)) + non_bg
        elif direction == 'left':
            for r in range(h):
                row = g[r]
                non_bg = [v for v in row if v != bg]
                g[r] = non_bg + [bg]*(w-len(non_bg))
        return [list(r) for r in g]

    for direction in ['down', 'up', 'right', 'left']:
        ok = True
        for ex in train:
            inp = [list(r) for r in ex['input']]
            expected = ex['output']
            pred = apply_gravity(inp, direction)
            if pred != [list(r) for r in expected]:
                ok = False
                break
        if ok:
            test_inp = [list(r) for r in task['test'][0]['input']]
            return apply_gravity(test_inp, direction), f'gravity_{direction}'
    return None


def solve_trim_border(task):
    """Remove uniform-color border rows/cols from input."""
    train = task['train']
    def trim(grid):
        g = np.array(grid)
        h, w = g.shape
        r0 = next((r for r in range(h) if len(set(g[r])) > 1), 0)
        r1 = next((r for r in range(h-1, -1, -1) if len(set(g[r])) > 1), h-1)
        c0 = next((c for c in range(w) if len(set(g[:, c])) > 1), 0)
        c1 = next((c for c in range(w-1, -1, -1) if len(set(g[:, c])) > 1), w-1)
        return g[r0:r1+1, c0:c1+1].tolist()
    try:
        for ex in train:
            pred = trim(ex['input'])
            if [list(r) for r in pred] != [list(r) for r in ex['output']]:
                return None
        return trim(task['test'][0]['input']), 'trim_border'
    except Exception:
        return None


def solve_object_count_recolor(task):
    """Recolor objects by their pixel count rank."""
    train = task['train']
    def count_recolor(inp, out=None):
        g = np.array(inp)
        bg = find_bg(g)
        labeled, n = ndimage.label(g != bg)
        if n == 0: return None
        obj_sizes = [(i+1, np.sum(labeled == i+1)) for i in range(n)]
        obj_sizes.sort(key=lambda x: -x[1])
        if out is not None:
            out_g = np.array(out)
            color_map = {}
            for rank, (label_id, _) in enumerate(obj_sizes):
                mask = labeled == label_id
                colors_out = set(int(x) for x in out_g[mask] if x != bg)
                if len(colors_out) == 1:
                    color_map[rank] = colors_out.pop()
                else:
                    return None
            return color_map
        return None
    try:
        color_maps = [count_recolor(ex['input'], ex['output']) for ex in train]
        if any(m is None for m in color_maps): return None
        if len(set(str(sorted(m.items())) for m in color_maps)) != 1: return None
        color_map = color_maps[0]
        g = np.array(task['test'][0]['input'])
        bg = find_bg(g)
        labeled, n = ndimage.label(g != bg)
        obj_sizes = [(i+1, np.sum(labeled == i+1)) for i in range(n)]
        obj_sizes.sort(key=lambda x: -x[1])
        result = g.copy()
        for rank, (label_id, _) in enumerate(obj_sizes):
            if rank in color_map:
                result[labeled == label_id] = color_map[rank]
        return result.tolist(), 'obj_count_recolor'
    except Exception:
        return None


def solve_all(task):
    strategies = [
        solve_nb_lut,
        solve_abstract,
        solve_colormap,
        solve_geometric,
        solve_gravity,         # NEW: gravity direction
        solve_scale,
        solve_tile,
        solve_tile_plan,
        solve_crop,
        solve_trim_border,     # NEW: trim uniform border
        solve_partition,
        solve_grid_separator,
        solve_boolean,
        solve_symmetry,
        solve_rowcol,
        solve_row_col_select,
        solve_flood,
        solve_minority_removal,
        solve_fill_enclosed,
        solve_per_object_transform,
        solve_subgrid_extract,
        solve_subgrid_by_scoring,
        solve_block_summary,
        solve_object_shape_extract,
        solve_object_count_recolor,  # NEW: rank-recolor objects
        solve_counting,
        solve_adj_recolor,
        solve_cellular_automaton,
        solve_enhanced_dt,
    ]

    # Collect top-2 distinct predictions for attempt_1/attempt_2
    results = []
    seen_grids = []

    def _add(result):
        grid = result[0] if isinstance(result, tuple) else result
        grid_key = str(grid)
        if grid_key not in seen_grids:
            seen_grids.append(grid_key)
            results.append(result)

    for fn in strategies:
        try:
            if fn == solve_nb_lut:
                for r in [1, 2]:
                    res = fn(task, radius=r)
                    if res:
                        _add(res)
                        if len(results) >= 2: break
            else:
                res = fn(task)
                if res:
                    _add(res)
        except Exception:
            continue
        if len(results) >= 2:
            break

    if not results:
        return None
    # Return as list of up to 2 (grid, method) tuples
    return results

In [ ]:
def _timeout_handler(signum, frame):
    raise TimeoutError('task timeout')

def solve_with_timeout(task, timeout=TASK_TIMEOUT):
    signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout)
    try:
        return solve_all(task)
    except TimeoutError:
        return None
    finally:
        signal.alarm(0)

In [ ]:
with open(TEST_FILE) as f:
    challenges = json.load(f)

task_ids = list(challenges.keys())
print(f"Tasks: {len(task_ids)}")

submission = {}
solved = 0
for i, task_id in enumerate(task_ids):
    task = challenges[task_id]
    test_cases = task["test"]
    task_preds = []
    for tc in test_cases:
        try:
            # Solve this specific test case; returns list of up to 2 (grid, method)
            single_task = {**task, "test": [tc]}
            result = solve_with_timeout(single_task, TASK_TIMEOUT)
            if result is not None:
                # solve_all now returns a list of up to 2 results
                if isinstance(result, list):
                    preds = result
                elif isinstance(result, tuple):
                    preds = [result]
                else:
                    preds = [(result, "unknown")]

                def _grid(r):
                    g = r[0] if isinstance(r, tuple) else r
                    return g if isinstance(g, list) else g.tolist()

                a1 = _grid(preds[0])
                a2 = _grid(preds[1]) if len(preds) > 1 else a1
                task_preds.append({"attempt_1": a1, "attempt_2": a2})
                solved += 1
            else:
                raise ValueError("None result")
        except Exception:
            h = len(tc["input"])
            w = len(tc["input"][0]) if h > 0 else 1
            blank = [[0]*w for _ in range(h)]
            task_preds.append({"attempt_1": blank, "attempt_2": blank})
    submission[task_id] = task_preds
    if (i+1) % 20 == 0:
        print(f"Progress: {i+1}/{len(task_ids)}, solved={solved}", flush=True)

print(f"Done: {solved}/{len(task_ids)} solved ({100*solved/len(task_ids):.1f}%)")

In [ ]:
with open(OUT_FILE, 'w') as f:
    json.dump(submission, f)

print(f"Saved: {OUT_FILE}")
print(f"Tasks: {len(submission)}")
# Validate
sample_key = list(submission.keys())[0]
assert 'attempt_1' in submission[sample_key][0], "Bad format!"
assert 'attempt_2' in submission[sample_key][0], "Bad format!"
print("Format check: ✓")
print("\n=== SUBMISSION READY ===")